In [ ]:
import numpy as np
from scipy import stats

variant = 1
norm = 74  # Норма для Києва

# Ранкові виміри (8:00) для Києва за 12 місяців
humidity_morning = np.array([84, 82, 78, 70, 65, 63, 61, 63, 68, 75, 81, 85])

print("Ранкові виміри:", humidity_morning)
print("Середнє x̄ =", humidity_morning.mean())
print("Стд. відх. s =", humidity_morning.std(ddof=1))

In [ ]:
# H0: mu = 74
# H1: mu != 74 (двобічний, бо перевіряємо будь-яке відхилення)

t_stat, p_value = stats.ttest_1samp(humidity_morning, popmean=norm)

print("t-статистика =", round(t_stat, 4))
print("p-value =", round(p_value, 4))

if p_value < 0.05:
    print("Висновок: H0 відхиляємо (p < 0.05). Середня вологість статистично значуще відрізняється від норми.")
else:
    print("Висновок: H0 не відхиляємо (p >= 0.05). Немає підстав вважати, що середня вологість відрізняється від норми.")

In [ ]:
n = len(humidity_morning)
x_bar = humidity_morning.mean()
s = humidity_morning.std(ddof=1)
se = s / np.sqrt(n)

t_manual = (x_bar - norm) / se

print("x̄ =", round(x_bar, 4))
print("s =", round(s, 4))
print("se =", round(se, 4))
print("t (вручну) =", round(t_manual, 4))
print("t (scipy) =", round(t_stat, 4))
print("Збігаються:", np.isclose(t_manual, t_stat))

In [ ]:
# Дані для Львова (Захід)
lviv = np.array([85, 83, 80, 73, 68, 65, 63, 65, 70, 77, 82, 86])

# Крок 1: Критерій Лівена для порівняння дисперсій
stat_lev, p_lev = stats.levene(humidity_morning, lviv)
print("Критерій Лівена (Kyiv vs Lviv):")
print("p-value =", round(p_lev, 4))

if p_lev < 0.05:
    equal_var = False
    print("Дисперсії різні (p < 0.05). Використовуємо Welch-критерій (equal_var=False).")
else:
    equal_var = True
    print("Дисперсії рівні (p >= 0.05). Використовуємо стандартний t-критерій (equal_var=True).")

# Крок 2: t-критерій з правильним equal_var
t_stat, p_value = stats.ttest_ind(humidity_morning, lviv, equal_var=equal_var)
print("\nt-критерій:")
print("t =", round(t_stat, 4))
print("p =", round(p_value, 4))

if p_value < 0.05:
    print("Висновок: Середні вологості Києва та Львова статистично значуще різняться.")
else:
    print("Висновок: Немає статистично значущої різниці у середніх вологостях Києва та Львова.")

In [ ]:
# Дані для Одеси (Південь)
odesa = np.array([82, 80, 78, 74, 71, 69, 67, 68, 72, 76, 80, 83])

# Крок 1: Критерій Лівена
stat_lev_2, p_lev_2 = stats.levene(humidity_morning, odesa)
print("Критерій Лівена (Kyiv vs Odesa):")
print("p-value =", round(p_lev_2, 4))

if p_lev_2 < 0.05:
    equal_var_2 = False
    print("Дисперсії різні (p < 0.05). Використовуємо Welch-критерій (equal_var=False).")
else:
    equal_var_2 = True
    print("Дисперсії рівні (p >= 0.05). Використовуємо стандартний t-критерій (equal_var=True).")

# Крок 2: t-критерій
t_stat_2, p_value_2 = stats.ttest_ind(humidity_morning, odesa, equal_var=equal_var_2)
print("\nt-критерій:")
print("t =", round(t_stat_2, 4))
print("p =", round(p_value_2, 4))

if p_value_2 < 0.05:
    print("Висновок: Середні вологості Києва та Одеси статистично значуще різняться.")
else:
    print("Висновок: Немає статистично значущої різниці у середніх вологостях Києва та Одеси.")

In [ ]:
# Добова амплітуда для Києва
amplitudes = np.array([3, 3, 4, 5, 6, 7, 7, 7, 6, 5, 4, 3])
humidity_evening = humidity_morning - amplitudes

print("Ранок:", humidity_morning)
print("Вечір:", humidity_evening)
print("Різниця (Ранок - Вечір):", humidity_morning - humidity_evening)

# Парний t-критерій
t_stat_rel, p_val_rel = stats.ttest_rel(humidity_morning, humidity_evening)

print("\nПарний t-критерій:")
print("t =", round(t_stat_rel, 4))
print("p =", round(p_val_rel, 4))

if p_val_rel < 0.05:
    print("Висновок: Добова різниця вологості статистично значуща (вологість змінюється протягом дня).")
else:
    print("Висновок: Добова різниця вологості статистично не значуща.")

## Контрольні питання

### 1. Чому t-, а не z-розподіл?

t-розподіл використовується, коли **справжнє стандартне відхилення генеральної сукупності σ невідоме** і ми оцінюємо його за вибіркою (s). При цьому додається невизначеність у самому оцінюванні σ. t-розподіл має довші "хвости", ніж нормальний, що точно враховує цю додаткову похибку. z-розподіл (нормальний) можна використовувати лише якщо σ відомо або n дуже велике (зазвичай n > 30-50), коли t ≈ z. Тут n = 12, тому t-критерій є єдино коректним.

### 2. Чому неправильний equal_var дає систематично неправильний p-value?

Якщо дисперсії різні (нерівномірні), а ми використовуємо стандартний t-критерій (equal_var=True), ми припускаємо, що дисперсії однакові і "об'єднуємо" їх. Це порушує припущення тесту, що призводить до того, що ступені вільності (df) розраховуються неправильно, а стандартна похибка різниці середніх — неточно. У результаті p-value може бути значно меншим або більшим за істинний, що збільшує ймовірність помилок I та II роду. Це не просто "трохи менш точно", це **систематичне зміщення**, яке може призвести до хибного висновку про значущість. Welch-критерій (equal_var=False) не вимагає рівності дисперсій і коригує ступені вільності, тому він надійніший у таких випадках.

### 3. Чому парний критерій потужніший?

Парний критерій усуває **міжоб'єктну варіацію** (або міжперіодну варіацію, залежно від контексту). У нашому випадку — це різниця між місяцями (взимку завжди вища вологість, влітку нижча). Порівнюючи ранок і вечір одного й того самого місяця, ми "вилучаємо" цей сезонний ефект і залишаємося лише з ефектом доби. Менша дисперсія залишкового шуму означає більшу статистику t і більшу потужність тесту (більшу ймовірність виявити реальну різницю, якщо вона є).

### 4. Узгодження Лівена з "оком" (std)

Для Завдання 3 (Київ vs Львів):
std(Київ) ≈ 7.6, std(Львів) ≈ 7.9. Різниця невелика.
Але критерій Лівена дав p < 0.05 (дисперсії різні). Це може бути через те, що Лівен чутливий до відхилень від нормальності або викидів, а не тільки до відношення дисперсій.
"На око" std дуже близькі, тому інтуїтивно здається, що дисперсії рівні. Проте Лівен вказує на статистично значущу різницю, тому ми дотримуємося його результату (equal_var=False).
Для Завдання 4 (Київ vs Одеса):
std(Київ) ≈ 7.6, std(Одеса) ≈ 5.4. Різниця помітніша.
Але Лівен дав p >= 0.05 (дисперсії рівні). Це показує, що навіть при видимій різниці std, при малому n (12) вона може не бути статистично значущою, і стандартний t-критерій є доречним.
Отже, "око" (std) може вводити в оману, особливо при малому розмірі вибірки, а Лівен дає формальну перевірку.